# Μάθημα 09 - Σχέδιο Σχημάτων Μεταγνώσης


## Ρύθμιση

Αυτό το σημειωματάριο παρουσιάζει το πρότυπο σχεδιασμού Μεταγνώσης χρησιμοποιώντας το Microsoft Agent Framework.

**Προαπαιτούμενα:**
- Διαμόρφωση ανάπτυξης Azure OpenAI μέσω μεταβλητών περιβάλλοντος
- Πιστοποίηση Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Τι είναι η Μεταγνώση;

Η μεταγνώση είναι **το να σκέφτεσαι για τη σκέψη**. Στο πλαίσιο των πρακτόρων Τεχνητής Νοημοσύνης, σημαίνει την κατασκευή πρακτόρων που μπορούν να:

- **Αυτοαντανακλώνται** στις δικές τους εξόδους και στη διαδικασία συλλογισμού
- **Εντοπίζουν σφάλματα** και ανακάμπτουν ομαλά αντί να αποτυγχάνουν σιωπηλά
- **Αξιολογούν** αν οι απαντήσεις τους είναι ολοκληρωμένες και χρήσιμες
- **Προσαρμόζουν** τη στρατηγική τους όταν μια αρχική προσέγγιση δεν λειτουργεί (π.χ., επιστρέφοντας σε ένα εφεδρικό σύστημα)

Ένας μεταγνωστικός πράκτορας δεν απαντά απλώς σε ερωτήσεις — παρακολουθεί την απόδοσή του και προσαρμόζεται αμέσως.


## Κύρια και Εφεδρικά Εργαλεία

Ένα κοινό πρότυπο μεταγνώσης είναι η **στρατηγική εφεδρείας**. Ο πράκτορας δοκιμάζει πρώτα ένα κύριο εργαλείο· αν αυτό αποτύχει (π.χ., σφάλμα 404), ο πράκτορας αναγνωρίζει την αποτυχία και μεταβαίνει διαφανώς σε ένα εφεδρικό εργαλείο.

Αυτό αντανακλά τα συστήματα στον πραγματικό κόσμο όπου οι κύριες υπηρεσίες μπορεί να μην είναι διαθέσιμες και οι πράκτορες πρέπει να αυτοδιαγνώσουν το πρόβλημα πριν επιλέξουν μια εναλλακτική διαδρομή.

Παρακάτω ορίζουμε δύο εργαλεία αναζήτησης πτήσεων:
- **Κύριο** — καλύπτει Παρίσι, Τόκιο και Βαρκελώνη
- **Εφεδρικό** — καλύπτει Βερολίνο, Σίδνεϊ και Νέα Υόρκη


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Πράκτορας Αυτοαναστοχασμού με Ανάκτηση Σφαλμάτων

Ο παρακάτω πράκτορας έχει εντολή να δοκιμάσει πρώτα το πρωτεύον σύστημα πτήσης, να αναγνωρίζει αποτυχίες και να μεταπίπτει διαφανώς στο εφεδρικό σύστημα. Μετά από κάθε απάντηση αξιολογεί σύντομα εάν απάντησε πλήρως στην ερώτηση του χρήστη.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Πρότυπο Αυτοαξιολόγησης

Μια άλλη όψη της μεταγνωσης είναι η **αυτοαξιολόγηση**: ένας ξεχωριστός παράγοντας (ή ο ίδιος ο παράγοντας σε δεύτερη φάση) αξιολογεί μια απάντηση για πληρότητα, ακρίβεια και χρησιμότητα.

Παρακάτω δημιουργούμε έναν παράγοντα `ResponseEvaluator` που βαθμολογεί τις απαντήσεις πρακτόρων ταξιδίων σε τρεις διαστάσεις.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Περίληψη

Σε αυτό το μάθημα μάθατε πώς να δημιουργείτε **μεταγνωστικούς πράκτορες** χρησιμοποιώντας το Microsoft Agent Framework:

- **Αυτοαντανάκλαση**: Πράκτορες που παρακολουθούν τη δική τους λογική και επικοινωνούν διαφανώς τι συνέβη.
- **Ανάκτηση από σφάλματα με εναλλακτικές λύσεις**: Ένα πρότυπο κύριου + εφεδρικού εργαλείου όπου ο πράκτορας εντοπίζει αποτυχίες (π.χ., σφάλματα 404) και αυτόματα δοκιμάζει μια εναλλακτική πηγή.
- **Αυτοαξιολόγηση**: Ένας ξεχωριστός πράκτορας αξιολογητής που βαθμολογεί τις απαντήσεις για πληρότητα, ακρίβεια και χρησιμότητα.

Αυτά τα πρότυπα καθιστούν τους πράκτορες πιο ανθεκτικούς, διαφανείς και αξιόπιστους — κρίσιμες ποιότητες για παραγωγικές αναπτύξεις.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
